# Final Model Approximation (lavaan)
**Approximating the Optimal Bifactor Structure via Confirmatory Factor Analysis (CFA)**

This notebook executes a Confirmatory Factor Analysis (CFA) to approximate the optimal latent structure identified in our Item Response Theory (IRT) analyses. Specifically, we transition from the `mirt` framework to `lavaan` to model the Bifactor structure of Neuroticism, featuring a general factor and two specific facets (Anxiety and Depression) identified by their two strongest-loading indicators respectively.

Similar to the previous stage of analysis, this notebook displays the **real analytical results derived from the official HCP-YA 2025 data release**. In strict adherence to the HCP Restricted Data Use Agreement, all presented outputs - such as standardized factor loadings and global fit indices - are aggregate statistics. No individual-level data or participant identifiers are revealed, ensuring full transparency of the results while maintaining participant confidentiality.

---
### Initialization

In [ ]:
# Set-Up
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)
from rpy2 import robjects as ro
from rpy2.robjects import pandas2ri, numpy2ri
from rpy2.robjects.packages import importr
pandas2ri.activate()
numpy2ri.activate()
ro.r('set.seed(123)')
%load_ext rpy2.ipython
importr('base')
importr('stats')
importr('lavaan')

# Load prepped data
# The notebook was executed with official HCP data before the file path was changed. 
# Please update the path below to your local version of the prepped official data.
neuroticism_df = pd.read_csv(r"YOUR\PATH\HERE\prepped_RESTRICTED_data_for_SEM.csv", dtype={'Subject': str})
neuro_items = neuroticism_df.drop(columns=['Subject'])
# Put into R
ro.globalenv['neuro_items'] = neuro_items

---
### Fitting the Confirmatory Factor Analysis

In [2]:
g_equation = " + ".join(neuro_items)

model_string = f"""
    g     =~ {g_equation}  # all items
    s_anx =~ a*NEORAW_01 + a*NEORAW_31
    s_dep =~ b*NEORAW_16 + b*NEORAW_46 
"""
ro.globalenv['model_string'] = model_string

In [3]:
%%R
lavaan_fit <- cfa(model_string, data = neuro_items, orthogonal=TRUE, std.lv=TRUE)
lavaan_fit@Fit@converged # check, model did converge

[1] TRUE


In [4]:
%%R
summary(lavaan_fit, fit.measures = TRUE, standardized=TRUE)

lavaan 0.6-20 ended normally after 18 iterations

  Estimator                                         ML
  Optimization method                           NLMINB
  Number of model parameters                        28
  Number of equality constraints                     2

  

Number of observations                          1198

Model Test User Model:
                                                      
  Test statistic                               269.410
  Degrees of freedom                                52
  P-value (Chi-square)                           0.000

Model Test Baseline Model:

  Test statistic                              3737.849
  Degrees of freedom                                66
  P-value                                        0.000

User Model versus Baseline Model:

  Comparative Fit Index (CFI)                    0.941
  Tucker-Lewis Index (TLI)                       0.925

Loglikelihood and Information Criteria:

  Loglikelihood user model (H0)             -18775.116
  Loglikelihood unrestricted model (H1)     -18640.410
                                                      
  Akaike (AIC)                               37602.231
  Bayesian (BIC)                             37734.530
  Sample-size adjusted Bayesian (SABIC)      3